| Name  | Registration Number |
|--------| -------------------|
| Akol Paul |  22/U/22453   |
| Beingana Jim Junior | 22/X/5243/PS |

## Project Overview:

CoCIS Real-Time Customer Care Service Voice Assistant is an AI-powered voice agent designed to serve students,Parents, staff, and visitors of the College of Computing and Information Sciences (CoCIS) at Makerere University in Uganda. The assistant is built to handle customer care inquiries through natural spoken language across multiple Ugandan languages, making it accessible to a diverse user base that may not be comfortable communicating in English alone.
The core challenge the project addresses is language inclusivity most existing voice assistants are English-only, leaving out millions of Ugandan speakers of languages like Acholi, Luganda, Lugbara, Ateso, Runyankole, and Swahili. This assistant bridges that gap by supporting real-time voice selection and multilingual response generation.
The project leverages the Sunbird AI SALT (Speech Annotations for Low-resource Transcription) dataset, a parallel speech corpus of 24,933 audio recordings across 7 Ugandan languages recorded in studio quality conditions. The dataset serves as the foundation for training the language identification, speech recognition, and voice synthesis components of the assistant.

The end goal is a deployable voice agent that can:

1. Detect which language a user is speaking
2. Transcribe and understand their query
3. Respond in the same language with a natural synthesized voice
4. Handle common CoCIS customer care tasks such as course inquiries, registration help, and departmental information, Tuition inquiries etc




In [ ]:
!pip install ydata-profiling --break-system-packages

1. Imports

In [ ]:
# check for dataset access token
import os
from google.colab import userdata

# load the dataset
from huggingface_hub import login
from datasets import load_dataset

# ML libraries
import io
import pandas as pd
import numpy as np
import soundfile as sf
from ydata_profiling import ProfileReport
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import librosa
import librosa.display
from wordcloud import WordCloud
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings('ignore')


2. Check for dataset access token

In [ ]:
HF_SUNBIRD_TOKEN = userdata.get('appauli_sunbird_access_token')

print(f"Token found: {HF_SUNBIRD_TOKEN is not None}")
print(f"Token preview: {HF_SUNBIRD_TOKEN[:15] if HF_SUNBIRD_TOKEN else 'None'}")

3. Load the Dataset

In [ ]:
login(token=HF_SUNBIRD_TOKEN)

#Load Multi-language configs
languages = ['studio-ach', 'studio-swa','studio-lug', 'studio-lgg', 'studio-teo', 'studio-eng', 'studio-nyn']

#create dictionary to store datasets
datasets_dict = {}
for lang in languages:
    print(f"Loading {lang}...")
    datasets_dict[lang] = load_dataset('sunbird/salt', lang)

print("\nAll loaded!")
print(list(datasets_dict.keys()))

4. Explore the Datasets

In [ ]:
# Check structure of one dataset
print(datasets_dict['studio-lug'])

In [ ]:
# See a sample entry
print(datasets_dict['studio-lug']['train'][0])

5. Convert all loaded datasets into DataFrames

In [ ]:
all_dfs = []

for lang, dataset in datasets_dict.items():
  # train, validation, test
    for split in dataset.keys():
        df = dataset[split].to_pandas()
        df['language'] = lang
        df['split'] = split
        all_dfs.append(df)

# Combine all into one master DataFrame
master_df = pd.concat(all_dfs, ignore_index=True)

print("All datasets merged successfully!")
print(f"Total shape: {master_df.shape}")

6. Explanatory Data Analysis

Step 1. Data Overview

In [ ]:
# Samples per language
print("Samples Per Language:")
samples_per_lang = master_df['language'].value_counts()
print(samples_per_lang)
print(f"\nTotal samples: {samples_per_lang.sum()}")

In [ ]:
# Column names and data types
print("Column Names and Data Types:")
print(master_df.dtypes)
print()

# Shape
print(f"Total Rows: {master_df.shape[0]}")
print(f"Total Columns: {master_df.shape[1]}")
print()

# Preview first few rows
print("First 3 rows:")
print(master_df.head(3))

In [ ]:
# Non-audio columns summary
non_audio_cols = [col for col in master_df.columns if col != 'audio']
print("Summary Statistics (text/categorical columns):")
print(master_df[non_audio_cols].describe(include='all'))

In [ ]:
# Audio column summary
print("Audio Column Summary:")
print(f"  Total audio samples: {master_df['audio'].notna().sum()}")
print(f"  Missing audio files: {master_df['audio'].isna().sum()}")

# Extract audio duration
print("\nExtracting audio durations...")

def get_duration(audio):
    try:
        return len(audio['array']) / audio['sampling_rate']
    except:
        return None

master_df['duration_seconds'] = master_df['audio'].apply(get_duration)
print("Duration extracted!")
print(master_df['duration_seconds'].describe())

In [ ]:
#The duration extraction returned 0 results, meaning the get_duration function failed silently on all rows
#inspect what the audio column looks like
sample = master_df['audio'].iloc[0]
print(type(sample))
print(sample)

In [ ]:
# Audio is stored as raw bytes (OggS format), not as a decoded array.
# That's why audio['array'] failed.
# Decode it

def get_duration(audio):
    try:
        audio_bytes = io.BytesIO(audio['bytes'])
        data, samplerate = sf.read(audio_bytes)
        return len(data) / samplerate
    except:
        return None

print("Extracting audio durations ...")
master_df['duration_seconds'] = master_df['audio'].apply(get_duration)
print("Duration extracted!")
print(master_df['duration_seconds'].describe())

In [ ]:
#Audio Duration per Language
print("Audio Duration Per Language (seconds)")
duration_per_lang = master_df.groupby('language')['duration_seconds'].describe()
print(duration_per_lang)

In [ ]:
# Speaker Count and gender Distribution

print("Speaker Count Per Language")

# Check what speaker/gender columns exist first
print("Available columns:", master_df.columns.tolist())
print()

# Speaker count
if 'speaker_id' in master_df.columns:
    speaker_count = master_df.groupby('language')['speaker_id'].nunique()
    print("Unique speakers per language:")
    print(speaker_count)

# Gender distribution
if 'gender' in master_df.columns:
    gender_dist = master_df.groupby(['language', 'gender']).size().unstack(fill_value=0)
    print("\nGender distribution per language:")
    print(gender_dist)

Speaker Count

All languages have only 1 unique speaker Acholi, Lugbara, Luganda, Runyankole, Swahili, and Ateso except English with 3 unique speakers

Limitation: Having only 1 speaker per language means:

The model learns one person's voice, accent, and speaking style
It may struggle to generalize to other speakers of the same language
There is no speaker diversity for most languages

Importantance to the Project

The voice assistant is designed to support multiple Ugandan languages with different voices
With only 1 speaker per language, voice variety within a language is very limited
English having 3 speakers gives it slightly more diversity

NOTE: No gender column exists in the dataset, so we can't analyze gender distribution

In [ ]:
# Text Length Distribution

print("Text Length Distribution")

master_df['char_length'] = master_df['text'].astype(str).apply(len)
master_df['word_count'] = master_df['text'].astype(str).apply(lambda x: len(x.split()))

print("Character length per language:")
print(master_df.groupby('language')['char_length'].describe())

print("\nWord count per language:")
print(master_df.groupby('language')['word_count'].describe())

# sample texts preview
print("\nSample texts per language:")
for lang in master_df['language'].unique():
    sample = master_df[master_df['language'] == lang]['text'].iloc[0]
    print(f"\n{lang}: {sample}")

All languages appear to be translating the same sentence about eggplants. meaning the dataset is a **parallel corpus**, the same sentences translated and recorded across all languages. This is very useful for the voice assistant because the linguistic content is consistent across languages.

Step 2 Data Quality Evaluation

In [ ]:
# Missing Values
print("Missing Values:")

null_counts = master_df.isnull().sum()
null_percent = (master_df.isnull().sum() / len(master_df)) * 100

null_summary = pd.DataFrame({
    'Null Count': null_counts,
    'Null Percentage (%)': null_percent.round(2)
})

print(null_summary)
print(f"\nTotal missing values: {null_counts.sum()}")

In [ ]:
# Duplicate Values

print("Duplicate Values Check:")

# Convert audio bytes to string for hashing
print("Converting audio column for duplicate checking...")
master_df['audio_hash'] = master_df['audio'].apply(lambda x: str(len(x['bytes'])))

# Now check all columns
total_duplicates = master_df.drop(columns=['audio']).duplicated().sum()
print(f"Total duplicate rows: {total_duplicates}")

# Check duplicate text entries
text_duplicates = master_df.duplicated(subset=['text']).sum()
print(f"Duplicate text entries: {text_duplicates}")

# Check duplicate IDs
id_duplicates = master_df.duplicated(subset=['id']).sum()
print(f"Duplicate IDs: {id_duplicates}")

# Check duplicate audio using hash
audio_duplicates = master_df.duplicated(subset=['audio_hash']).sum()
print(f"Duplicate audio files (by size): {audio_duplicates}")

# Show duplicate texts if any
if text_duplicates > 0:
    print("\nSample duplicate texts:")
    dupes = master_df[master_df.duplicated(subset=['text'], keep=False)]
    print(dupes[['language', 'text', 'id']].head(10))
else:
    print("\nNo duplicate texts found!")

if id_duplicates > 0:
    print("\nSample duplicate IDs:")
    dupes = master_df[master_df.duplicated(subset=['id'], keep=False)]
    print(dupes[['language', 'id']].head(10))
else:
    print("No duplicate IDs found!")

In [ ]:
# Corrupted Audio checks

print("Corrupted Audio Check:")

def check_audio(audio):
    try:
        audio_bytes = io.BytesIO(audio['bytes'])
        data, samplerate = sf.read(audio_bytes)
        if len(data) == 0:
            return 'empty'
        duration = len(data) / samplerate
        if duration < 0.1:
            return 'too_short'
        # np.abs(data), converts all negative sound wave values to positive
        #  np.max(...), finds the loudest point in the audio
        if np.max(np.abs(data)) == 0:
            return 'silent'
        return 'ok'
    except:
        return 'corrupted'

print("Checking all audio files...")
master_df['audio_status'] = master_df['audio'].apply(check_audio)

print("\nAudio Status Summary:")
print(master_df['audio_status'].value_counts())

print("\nAudio Status per Language:")
print(master_df.groupby(['language', 'audio_status']).size().unstack(fill_value=0))

In [ ]:
# Find and examine the problematic file
problematic = master_df[master_df['audio_status'] == 'too_short']
print(problematic[['id', 'language', 'text', 'duration_seconds', 'audio_status']])

Observations
The text appears to be a full sentence but its just short.

In [ ]:
# Inconsistent Labels examination

print("Inconsistent Label Check:")

print("Unique values in audio_language:")
print(master_df['audio_language'].unique())

print("\nUnique values in language:")
print(master_df['language'].unique())

print("\nUnique values in is_studio:")
print(master_df['is_studio'].unique())

print("\nUnique values in split:")
print(master_df['split'].unique())

print("\nSplit distribution per language:")
print(master_df.groupby(['language', 'split']).size().unstack(fill_value=0))

print("\nChecking audio_language vs language consistency...")
mismatch = master_df[master_df['audio_language'] != master_df['language'].str.replace('studio-', '')]
print(f"Mismatched labels: {len(mismatch)}")
if len(mismatch) > 0:
    print(mismatch[['language', 'audio_language']].head(10))

audio_language vs language column

audio_language uses short codes: ach, swa, lug, lgg, teo, eng, nyn
language uses full names: studio-ach, studio-swa, studio-lug etc.
These are the same languages just formatted differently no real mismatch

is_studio column

Only value is True for all 24,933 samples
Confirms every single recording is a studio quality recording
No field recordings or noisy audio mixed in
consistent audio quality throughout

split column

Three splits exist: train, dev, test
Note: dev is the validation set (development set)
All languages have all three splits no language is missing a split

In [ ]:
# Imbalanced Data checks

print("Imbalanced Data Check")

# Language distribution
lang_counts = master_df['language'].value_counts()
lang_percent = (lang_counts / len(master_df) * 100).round(2)

imbalance_summary = pd.DataFrame({
    'Count': lang_counts,
    'Percentage (%)': lang_percent
})

print("Language Distribution:")
print(imbalance_summary)

# Check imbalance ratio
max_count = lang_counts.max()
min_count = lang_counts.min()
imbalance_ratio = round(max_count / min_count, 2)

print(f"\nLargest class:  {lang_counts.idxmax()} ({max_count} samples)")
print(f"Smallest class: {lang_counts.idxmin()} ({min_count} samples)")
print(f"Imbalance ratio: {imbalance_ratio}:1")

if imbalance_ratio > 3:
    print("Significant class imbalance detected!")
else:
    print("Class imbalance is acceptable")

# Split imbalance check
print("\nSplit Distribution:")
print(master_df['split'].value_counts())
split_percent = (master_df['split'].value_counts() / len(master_df) * 100).round(2)
print("\nSplit Percentage:")
print(split_percent)

Language Imbalance
The imbalance ratio is 3.02:1 which just crossed the threshold of concern:

studio-ach (Acholi) is the largest class with 5,000 samples (20.05%)
studio-lgg (Lugbara) is the smallest with only 1,654 samples (6.63%)
The difference is about 3,346 samples between largest and smallest

This means the model will naturally learn Acholi better than Lugbara simply because it has 3x more data to learn from.

In [ ]:
# Correlation checks
print("Correlation Check")

# Select only numeric columns
numeric_cols = master_df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric columns available: {numeric_cols}")

# Compute correlation matrix
corr_matrix = master_df[numeric_cols].corr()
print("\nCorrelation Matrix:")
print(corr_matrix.round(3))

# Find highly correlated pairs
print("\nHighly Correlated Pairs (correlation > 0.8):")
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr_val = corr_matrix.iloc[i, j]
        if abs(corr_val) > 0.8:
            high_corr_pairs.append({
                'Feature 1': corr_matrix.columns[i],
                'Feature 2': corr_matrix.columns[j],
                'Correlation': round(corr_val, 3)
            })

if high_corr_pairs:
    print(pd.DataFrame(high_corr_pairs))
else:
    print("No highly correlated pairs found!")

Correlation Matrix Evaluation

No pairs exceeded the 0.8 threshold, but there are some notable relationships:

**char_length vs word_count: 0.785**

This is the strongest correlation in the dataset

longer texts in characters naturally have more words
At 0.785 it is just below the 0.8 threshold but still very high
Recommendation
Consider keeping only one of these features to avoid redundancy

**duration_seconds vs word_count: 0.330**

Moderate positive correlation
More words = longer audio
Not strong enough to be a concern


**duration_seconds vs char_length: 0.298**

Similar moderate relationship
Longer text = longer audio recording
Also expected and not a concern

**id vs everything: negative correlations**

id has small negative correlations with all features
This is meaningless, ID is just a row number, not a real feature
Recommendation
Should be dropped before model training

**speaker_id correlations: all near zero**

speaker_id has very weak correlations with everything
With only 1 speaker per language for most languages, very little useful information

In [ ]:
# Redundant Features Check

print("Redundant Features Check:")

# Check if audio_language and language carry same info
print("audio_language unique values:", master_df['audio_language'].nunique())
print("language unique values:", master_df['language'].nunique())

# Check if they map 1:1
mapping = master_df.groupby('audio_language')['language'].unique()
print("\naudio_language to language mapping:")
print(mapping)

# Check is_studio - is it constant?
print(f"\nis_studio unique values: {master_df['is_studio'].unique()}")
print(f"is_studio is constant: {master_df['is_studio'].nunique() == 1}")

# Check speaker_id - is it useful?
print(f"\nspeaker_id unique values: {master_df['speaker_id'].nunique()}")
print(f"speaker_id distribution:")
print(master_df['speaker_id'].value_counts())

# char_length vs word_count relationship
corr_text = master_df['char_length'].corr(master_df['word_count'])
print(f"\nCorrelation between char_length and word_count: {corr_text:.3f}")
if abs(corr_text) > 0.8:
    print("char_length and word_count are highly correlated - consider keeping only one!")

**audio_language vs language: Redundant**

Both columns represent exactly the same 7 languages
audio_language uses short codes: ach, eng, lgg, lug, nyn, swa, teo
language uses full names: studio-ach, studio-eng etc.
They map perfectly 1:1

Recommendation
one column should be dropped

**is_studio: Redundant**
Only has one value: True for all 24,912 samples
A column with only one value carries zero information
It cannot help any model learn anything

Recommendation
drop is_studio completely

**speaker_id**
Speaker_id does NOT map 1:1 with language
There are 7 unique speaker IDs matching 7 languages
But earlier we saw English had 3 speakers, this means speaker 246 has 4973 samples which likely spans multiple languages
speaker_id is a numerical ID not a language label so it carries different information than language

**char_length vs word_count: 0.785**
As noted in correlation check, these are highly related

Recommendation
keep word_count, drop char_length since word count is more meaningful for speech analysis

In [ ]:
# Noise detection

print("Noise Detection")

# Duration outliers using IQR method
Q1 = master_df['duration_seconds'].quantile(0.25)
Q3 = master_df['duration_seconds'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

duration_outliers = master_df[
    (master_df['duration_seconds'] < lower_bound) |
    (master_df['duration_seconds'] > upper_bound)
]

print(f"Duration IQR: {IQR:.2f}")
print(f"Lower bound: {lower_bound:.2f}")
print(f"Upper bound: {upper_bound:.2f}")
print(f"Duration outliers: {len(duration_outliers)}")
print(f"Outliers per language:")
print(duration_outliers['language'].value_counts())

# Text length outliers
Q1_text = master_df['word_count'].quantile(0.25)
Q3_text = master_df['word_count'].quantile(0.75)
IQR_text = Q3_text - Q1_text

lower_text = Q1_text - 1.5 * IQR_text
upper_text = Q3_text + 1.5 * IQR_text

text_outliers = master_df[
    (master_df['word_count'] < lower_text) |
    (master_df['word_count'] > upper_text)
]

print(f"\nWord count outliers: {len(text_outliers)}")
print(f"Outliers per language:")
print(text_outliers['language'].value_counts())

# Very short texts
short_texts = master_df[master_df['word_count'] <= 2]
print(f"\nVery short texts (2 words or less): {len(short_texts)}")
print(short_texts[['language', 'text', 'word_count']].head(10))

Duration Outliers: 933 samples
The IQR bounds are:
Lower bound: -0.26, negative duration is impossible, so effectively 0
Upper bound: 9.77 seconds anything above this is an outlier

studio-lgg (Lugbara) dominates with 782 outliers this confirms what we saw earlier where Lugbara has unusually long recordings averaging 17.5 seconds. Almost all of its 1,654 samples are outliers by duration!

Word Count Outliers: 438 samples
studio-ach has 220 word count outliers unusually long sentences
studio-lgg has 215 consistent with its longer text pattern
studio-swa and studio-nyn have very few outliers


Very Short Texts: 102 samples

102 samples have 2 words or less
Most are in studio-ach (Acholi) short phrases like "Gicoyo latela", "Pingo imer?"
One sample in studio-swa has only 1 word "Sanitiser" this is clearly not a proper sentence

Conclusion
These short texts may be greetings, questions or incomplete sentences

Step 3 Feature Assessment and Visualization

Set up

In [ ]:
# Language mappings
LANG_LABELS = {
    'studio-ach': 'Acholi',
    'studio-eng': 'English',
    'studio-lgg': 'Lugbara',
    'studio-lug': 'Luganda',
    'studio-nyn': 'Runyankole',
    'studio-swa': 'Swahili',
    'studio-teo': 'Ateso'
}

LANG_COLORS = {
    'studio-ach': '#E63946',
    'studio-eng': '#2196F3',
    'studio-lgg': '#FF9800',
    'studio-lug': '#4CAF50',
    'studio-nyn': '#9C27B0',
    'studio-swa': '#00BCD4',
    'studio-teo': '#F06292'
}

LANGUAGES   = list(LANG_LABELS.keys())
COLOR_LIST  = [LANG_COLORS[l] for l in LANGUAGES]
LABEL_LIST  = [LANG_LABELS[l] for l in LANGUAGES]

# Global style
BG      = '#0F1117'
PANEL   = '#1A1D27'
GRID    = '#2D3147'
TEXT    = '#E0E0E0'
SUBTEXT = '#9E9E9E'

plt.rcParams.update({
    'figure.facecolor'  : BG,
    'axes.facecolor'    : PANEL,
    'axes.edgecolor'    : GRID,
    'axes.labelcolor'   : TEXT,
    'axes.titlecolor'   : TEXT,
    'axes.titlesize'    : 13,
    'axes.labelsize'    : 11,
    'xtick.color'       : SUBTEXT,
    'ytick.color'       : SUBTEXT,
    'text.color'        : TEXT,
    'grid.color'        : GRID,
    'grid.linestyle'    : '--',
    'grid.alpha'        : 0.5,
    'font.family'       : 'DejaVu Sans',
    'legend.facecolor'  : PANEL,
    'legend.edgecolor'  : GRID,
})

# save & show
def save_show(fig, filename):
    fig.savefig(f'/mnt/user-data/outputs/{filename}',
                dpi=150, bbox_inches='tight', facecolor=BG)
    plt.show()
    print(f"Saved {filename}")

# decode audio bytes
def decode_audio(audio_dict):
    audio_bytes = io.BytesIO(audio_dict['bytes'])
    data, sr = sf.read(audio_bytes)
    if data.ndim > 1:
        data = data.mean(axis=1)
    return data.astype(np.float32), sr

# Create output directory
os.makedirs('/mnt/user-data/outputs', exist_ok=True)

print("Setup complete, ready to plot!")

A: Univariate Analysis

1. Bar Chart of Sample Count per Language

Shows how many audio samples exist for each of the 7 languages.
it reveals class imbalance eg Acholi has 5,000 samples while Lugbara has only 1,654.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(BG)

lang_counts = master_df['language'].value_counts().reindex(LANGUAGES)
lang_names  = [LANG_LABELS[l] for l in lang_counts.index]
bars = ax.bar(lang_names, lang_counts.values,
              color=COLOR_LIST, edgecolor='white', linewidth=0.5, width=0.6)

# Value labels on bars
for bar, val in zip(bars, lang_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 30,
            f'{val:,}', ha='center', va='bottom',
            fontsize=10, color=TEXT, fontweight='bold')

ax.set_title('A Bar Grapg Showing a Sample Count per Language', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Language', labelpad=10)
ax.set_ylabel('Number of Samples', labelpad=10)
ax.tick_params(axis='x', rotation=25)
ax.grid(axis='y', alpha=0.4)
ax.set_axisbelow(True)

# Category label
ax.text(0.01, 0.97, 'UNIVARIATE Distribution',
        transform=ax.transAxes, fontsize=9,
        color=SUBTEXT, va='top')

plt.tight_layout()
save_show(fig, 'bar_graph_of_sample_count.png')

2.  Histogram Showing Audio Duration Distribution

Shows the spread of audio clip lengths across the entire dataset.  
it shows that most clips are between 3 to 6 seconds but Lugbara has very long clips up to 50 seconds.

Importance
Shows Consistency

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(BG)

ax.hist(master_df['duration_seconds'],
        bins=60, color='#00BCD4',
        edgecolor='white', linewidth=0.3, alpha=0.85)

mean_val   = master_df['duration_seconds'].mean()
median_val = master_df['duration_seconds'].median()

ax.axvline(mean_val,   color='#FF9800', linestyle='--',
           linewidth=2, label=f'Mean: {mean_val:.2f}s')
ax.axvline(median_val, color='#E63946', linestyle='--',
           linewidth=2, label=f'Median: {median_val:.2f}s')

ax.set_title('Histogram Showing Audio Duration Distribution', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Duration (seconds)', labelpad=10)
ax.set_ylabel('Frequency', labelpad=10)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.4)
ax.set_axisbelow(True)
ax.text(0.01, 0.97, 'UNIVARIATE — Distribution',
        transform=ax.transAxes, fontsize=9, color=SUBTEXT, va='top')

plt.tight_layout()
save_show(fig, 'audio_duration_distribution_histogram.png')

3. Histogram of Text Length (Word Count) Distribution

Shows how many words are in each sentence.
Short texts with 1 to 2 words may be noise or incomplete sentences. There are 102 very short texts.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(BG)

ax.hist(master_df['word_count'],
        bins=40, color='#9C27B0',
        edgecolor='white', linewidth=0.3, alpha=0.85)

mean_wc   = master_df['word_count'].mean()
median_wc = master_df['word_count'].median()

ax.axvline(mean_wc,   color='#FF9800', linestyle='--',
           linewidth=2, label=f'Mean: {mean_wc:.1f} words')
ax.axvline(median_wc, color='#E63946', linestyle='--',
           linewidth=2, label=f'Median: {median_wc:.1f} words')

ax.set_title('Histogram Showing Text Length (Word Count) Distribution',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Word Count', labelpad=10)
ax.set_ylabel('Frequency', labelpad=10)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.4)
ax.set_axisbelow(True)
ax.text(0.01, 0.97, 'UNIVARIATE Distribution',
        transform=ax.transAxes, fontsize=9, color=SUBTEXT, va='top')

plt.tight_layout()
save_show(fig, 'wordcount_histogram.png')

4.  Bar Chart Showing Speaker Count per Language

Shows how many unique speakers recorded audio for each language.
It's important for the voice agent because a model trained on only 1 speaker per language will struggle to generalize to new voices.
The dataset has only 1 speaker for 6 out of 7 languages which is a significant limitation.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(BG)

spk_counts = master_df.groupby('language')['speaker_id'].nunique().reindex(LANGUAGES)
bars = ax.bar([LANG_LABELS[l] for l in spk_counts.index],
              spk_counts.values,
              color=COLOR_LIST, edgecolor='white', linewidth=0.5, width=0.5)

for bar, val in zip(bars, spk_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.05,
            str(val), ha='center', va='bottom',
            fontsize=12, fontweight='bold', color=TEXT)

ax.set_title('Bar Chart Showing Speaker Count per Language',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Language', labelpad=10)
ax.set_ylabel('Number of Unique Speakers', labelpad=10)
ax.tick_params(axis='x', rotation=25)
ax.set_ylim(0, spk_counts.max() + 1)
ax.grid(axis='y', alpha=0.4)
ax.set_axisbelow(True)
ax.text(0.01, 0.97, 'UNIVARIATE Distribution',
        transform=ax.transAxes, fontsize=9, color=SUBTEXT, va='top')

plt.tight_layout()
save_show(fig, 'speaker_count.png')

5. Bar Chart of Train/Val/Test Split Sizes

Shows the absolute size of each data split.

It's important to confirm there is enough data in each split for meaningful training, validation, and evaluation.
The dataset has a very heavy train split at 96% which is common in speech datasets but leaves very little for validation at 2%.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor(BG)

split_counts  = master_df['split'].value_counts().reindex(['train', 'dev', 'test'])
split_colors  = ['#4CAF50', '#FF9800', '#E63946']
split_labels  = ['Train', 'Dev (Val)', 'Test']

bars = ax.bar(split_labels, split_counts.values,
              color=split_colors, edgecolor='white',
              linewidth=0.5, width=0.5)

for bar, val in zip(bars, split_counts.values):
    pct = val / split_counts.sum() * 100
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 100,
            f'{val:,}\n({pct:.1f}%)',
            ha='center', va='bottom',
            fontsize=11, fontweight='bold', color=TEXT)

ax.set_title('Bar Graph Showing Train / Dev / Test Split Sizes',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Split', labelpad=10)
ax.set_ylabel('Number of Samples', labelpad=10)
ax.grid(axis='y', alpha=0.4)
ax.set_axisbelow(True)
ax.text(0.01, 0.97, 'UNIVARIATE Distribution',
        transform=ax.transAxes, fontsize=9, color=SUBTEXT, va='top')

plt.tight_layout()
save_show(fig, 'split_sizes.png')

6. Pie Chart Showing Language Share


Shows each language as a percentage of the total dataset.
A quick visual way to communicate dataset composition to stakeholders.  Acholi and Ateso dominate at roughly 40% combined while Lugbara is underrepresented at only 6.6%.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
fig.patch.set_facecolor(BG)

lang_counts = master_df['language'].value_counts().reindex(LANGUAGES)
explode     = [0.03] * len(LANGUAGES)

wedges, texts, autotexts = ax.pie(
    lang_counts.values,
    labels      = LABEL_LIST,
    colors      = COLOR_LIST,
    explode     = explode,
    autopct     = '%1.1f%%',
    startangle  = 140,
    textprops   = {'color': TEXT, 'fontsize': 11},
    wedgeprops  = {'edgecolor': BG, 'linewidth': 2}
)
for at in autotexts:
    at.set_fontsize(9)
    at.set_color('white')
    at.set_fontweight('bold')

ax.set_title('A Pie Chart Showing Each Language Distribution in the Dataset',
             fontsize=15, fontweight='bold', pad=20)
ax.text(0.01, 0.01, 'UNIVARIATE Distribution',
        transform=ax.transAxes, fontsize=9, color=SUBTEXT)

plt.tight_layout()
save_show(fig, 'language_distribution_pie_chart.png')

7. Density Plot of Audio Duration

shows where most durations are concentrated. More informative than a histogram because it shows the shape of the distribution clearly whether it is normal, skewed, or multimodal.
Shows a right skew due to Lugbara's long recordings.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(BG)

master_df['duration_seconds'].plot.kde(
    ax=ax, color='#00BCD4', linewidth=2.5)

ax.fill_between(
    ax.lines[0].get_xdata(),
    ax.lines[0].get_ydata(),
    alpha=0.2, color='#00BCD4')

ax.axvline(master_df['duration_seconds'].mean(),
           color='#FF9800', linestyle='--', linewidth=2,
           label=f"Mean: {master_df['duration_seconds'].mean():.2f}s")

ax.set_title('Audio Duration Density Plot',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Duration (seconds)', labelpad=10)
ax.set_ylabel('Density', labelpad=10)
ax.legend(fontsize=10)
ax.grid(alpha=0.4)
ax.set_axisbelow(True)
ax.text(0.01, 0.97, 'UNIVARIATE Density',
        transform=ax.transAxes, fontsize=9, color=SUBTEXT, va='top')

plt.tight_layout()
save_show(fig, 'duration_density.png')

8. Density Plot of Word Count

Shows whether most sentences are short, long, or evenly distributed.
Most texts are between 6 to 13 words.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(BG)

master_df['word_count'].plot.kde(
    ax=ax, color='#9C27B0', linewidth=2.5)

ax.fill_between(
    ax.lines[0].get_xdata(),
    ax.lines[0].get_ydata(),
    alpha=0.2, color='#9C27B0')

ax.axvline(master_df['word_count'].mean(),
           color='#FF9800', linestyle='--', linewidth=2,
           label=f"Mean: {master_df['word_count'].mean():.1f} words")

ax.set_title('Word Count Density Plot',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Word Count', labelpad=10)
ax.set_ylabel('Density', labelpad=10)
ax.legend(fontsize=10)
ax.grid(alpha=0.4)
ax.set_axisbelow(True)
ax.text(0.01, 0.97, 'UNIVARIATE Density',
        transform=ax.transAxes, fontsize=9, color=SUBTEXT, va='top')

plt.tight_layout()
save_show(fig, 'wordcount_density.png')

9. Count Plot Showing Split Distribution

A seaborn count plot showing train, dev, and test split sizes with exact counts.  
Important to confirm no split is empty or too small for reliable evaluation.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor(BG)

split_order  = ['train', 'dev', 'test']
split_colors = ['#4CAF50', '#FF9800', '#E63946']

sns.countplot(data=master_df, x='split',
              order=split_order,
              palette=dict(zip(split_order, split_colors)),
              edgecolor='white', linewidth=0.5, ax=ax)

for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}',
                (p.get_x() + p.get_width()/2, p.get_height() + 50),
                ha='center', va='bottom',
                fontsize=11, fontweight='bold', color=TEXT)

ax.set_xticklabels(['Train', 'Dev (Val)', 'Test'])
ax.set_title('Split Distribution Count Plot',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Split', labelpad=10)
ax.set_ylabel('Count', labelpad=10)
ax.grid(axis='y', alpha=0.4)
ax.set_axisbelow(True)
ax.text(0.01, 0.97, 'UNIVARIATE Count Plot',
        transform=ax.transAxes, fontsize=9, color=SUBTEXT, va='top')

plt.tight_layout()
save_show(fig, 'split_countplot.png')

10. Count Plot Showing Language Distribution


A seaborn count plot of sample counts per language.
Useful for presentations and reports where a clean consistent style is needed.
Reinforces the class imbalance finding.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(BG)

lang_order  = master_df['language'].value_counts().index.tolist()
lang_labels_order = [LANG_LABELS[l] for l in lang_order]
lang_colors_order = [LANG_COLORS[l] for l in lang_order]

temp_df = master_df.copy()
temp_df['lang_label'] = temp_df['language'].map(LANG_LABELS)

sns.countplot(data=temp_df, x='lang_label',
              order=lang_labels_order,
              palette=dict(zip(lang_labels_order, lang_colors_order)),
              edgecolor='white', linewidth=0.5, ax=ax)

for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}',
                (p.get_x() + p.get_width()/2, p.get_height() + 30),
                ha='center', va='bottom',
                fontsize=10, fontweight='bold', color=TEXT)

ax.set_title('Language Distribution Count Plot',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Language', labelpad=10)
ax.set_ylabel('Count', labelpad=10)
ax.tick_params(axis='x', rotation=25)
ax.grid(axis='y', alpha=0.4)
ax.set_axisbelow(True)
ax.text(0.01, 0.97, 'UNIVARIATE Count Plot',
        transform=ax.transAxes, fontsize=9, color=SUBTEXT, va='top')

plt.tight_layout()
save_show(fig, 'language_countplot.png')

11.  Cumulative Distribution Audio Duration

Shows what percentage of clips fall below a given duration threshold.
Approximately 90% of clips are under 10 seconds except for Lugbara.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(BG)

sorted_dur = np.sort(master_df['duration_seconds'])
cdf        = np.arange(1, len(sorted_dur)+1) / len(sorted_dur)

ax.plot(sorted_dur, cdf, color='#00BCD4', linewidth=2.5)
ax.fill_between(sorted_dur, cdf, alpha=0.15, color='#00BCD4')

p50 = np.percentile(sorted_dur, 50)
p90 = np.percentile(sorted_dur, 90)
ax.axvline(p50, color='#FF9800', linestyle='--', linewidth=1.5,
           label=f'50th pct: {p50:.1f}s')
ax.axvline(p90, color='#E63946', linestyle='--', linewidth=1.5,
           label=f'90th pct: {p90:.1f}s')

ax.set_title('A Pot of Cumulative Distribution of Audio Duration',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Duration (seconds)', labelpad=10)
ax.set_ylabel('Cumulative Proportion', labelpad=10)
ax.legend(fontsize=10)
ax.grid(alpha=0.4)
ax.set_axisbelow(True)
ax.text(0.01, 0.97, 'UNIVARIATE CDF',
        transform=ax.transAxes, fontsize=9, color=SUBTEXT, va='top')

plt.tight_layout()
save_show(fig, 'duration_cdf.png')

12. Energy/Amplitude Distribution

Shows the loudness distribution of audio samples per language.
It's important for audio quality assessment.
If one language has consistently lower energy it may indicate quieter recordings or microphone issues.
Consistent energy levels across languages means more reliable synthesis.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(BG)

# Sample one audio file per language and compute RMS energy
energies = []
for lang in LANGUAGES:
    subset = master_df[master_df['language'] == lang]
    sample_audio = subset.iloc[0]['audio']
    data, sr = decode_audio(sample_audio)
    rms = librosa.feature.rms(y=data)[0]
    for val in rms:
        energies.append({'language': LANG_LABELS[lang], 'rms': val,
                         'color': LANG_COLORS[lang]})

energy_df = pd.DataFrame(energies)

for lang_name in energy_df['language'].unique():
    subset = energy_df[energy_df['language'] == lang_name]
    color  = subset['color'].iloc[0]
    subset['rms'].plot.kde(ax=ax, label=lang_name,
                           color=color, linewidth=2)

ax.set_title('Energy (RMS Amplitude) Distribution per Language',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('RMS Energy', labelpad=10)
ax.set_ylabel('Density', labelpad=10)
ax.legend(fontsize=9)
ax.grid(alpha=0.4)
ax.set_axisbelow(True)
ax.text(0.01, 0.97, 'UNIVARIATE Distribution',
        transform=ax.transAxes, fontsize=9, color=SUBTEXT, va='top')

plt.tight_layout()
save_show(fig, 'energy_distribution.png')

B. MULTIVARIATE ANALYSIS

1. Grouped Bargraph Showing Split Sizes per Language

Shows train, dev, and test counts side by side for each language.

 Shows whether splits are proportionally distributed across languages.
 If one language has very few test samples evaluation results for that language will be unreliable.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor(BG)

split_lang = master_df.groupby(['language', 'split']).size().unstack(fill_value=0)
split_lang  = split_lang.reindex(LANGUAGES)
x           = np.arange(len(LANGUAGES))
width       = 0.25
split_cols  = {'train': '#4CAF50', 'dev': '#FF9800', 'test': '#E63946'}

for i, (split, color) in enumerate(split_cols.items()):
    if split in split_lang.columns:
        bars = ax.bar(x + i*width, split_lang[split],
                      width, label=split.capitalize(),
                      color=color, edgecolor='white', linewidth=0.4)

ax.set_xticks(x + width)
ax.set_xticklabels(LABEL_LIST, rotation=25)
ax.set_title('Grouped Bargraph Showing Split Distribution per Language',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Language', labelpad=10)
ax.set_ylabel('Number of Samples', labelpad=10)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.4)
ax.set_axisbelow(True)
ax.text(0.01, 0.97, 'MULTIVARIATE Comparison',
        transform=ax.transAxes, fontsize=9, color=SUBTEXT, va='top')

plt.tight_layout()
save_show(fig, 'split_per_language.png')

2. Box Plot of Audio Duration per Language


Shows the median, spread, and outliers of audio duration for each language side by side.
It clearly exposes Lugbara as a major outlier with a median around 9 seconds compared to 3 to 5 seconds for all other languages. Outlier dots above the whiskers represent individual problematic recordings.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor(BG)

data_by_lang = [master_df[master_df['language'] == l]['duration_seconds'].values
                for l in LANGUAGES]

bp = ax.boxplot(data_by_lang, patch_artist=True,
                medianprops=dict(color='white', linewidth=2.5),
                whiskerprops=dict(color=SUBTEXT, linewidth=1.2),
                capprops=dict(color=SUBTEXT, linewidth=1.2),
                flierprops=dict(marker='o', markersize=3,
                                markerfacecolor=SUBTEXT, alpha=0.5))

for patch, color in zip(bp['boxes'], COLOR_LIST):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)

ax.set_xticks(range(1, len(LANGUAGES)+1))
ax.set_xticklabels(LABEL_LIST, rotation=25)
ax.set_title('Box Plot Showing Audio Duration Spread per Language',
             fontsize=15, fontweight='bold', pad=15)
ax.set_ylabel('Duration (seconds)', labelpad=10)
ax.grid(axis='y', alpha=0.4)
ax.set_axisbelow(True)
ax.text(0.01, 0.97, 'MULTIVARIATE — Comparison',
        transform=ax.transAxes, fontsize=9, color=SUBTEXT, va='top')

plt.tight_layout()
save_show(fig, 'audio_duration_boxplot.png')

3. Violin Plot of Word Count per Language

Combines a box plot with a density curve showing the full shape of word count distribution per language. More informative than a box plot alone because it shows whether each language has a single peak or multiple clusters.
It reveals that Lugbara and Acholi have wider distributions meaning more sentence length variety.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor(BG)

temp_df = master_df.copy()
temp_df['lang_label'] = temp_df['language'].map(LANG_LABELS)

# map colors to label names
label_colors = {LANG_LABELS[k]: v for k, v in LANG_COLORS.items()}

sns.violinplot(data=temp_df, x='lang_label', y='word_count',
               order=LABEL_LIST,
               palette=label_colors,
               inner='quartile',
               ax=ax)

ax.set_title('Violin Plot Showing Word Count Distribution per Language',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Language', labelpad=10)
ax.set_ylabel('Word Count', labelpad=10)
ax.tick_params(axis='x', rotation=25)
ax.grid(axis='y', alpha=0.4)
ax.set_axisbelow(True)
ax.text(0.01, 0.97, 'MULTIVARIATE Comparison',
        transform=ax.transAxes, fontsize=9, color=SUBTEXT, va='top')

plt.tight_layout()
save_show(fig, 'wordcount_violin.png')

4. Bar Chart Showing Average Duration per Language

Shows the mean audio duration per language as a simple comparison bar. The most direct way to confirm that Lugbara recordings are longer than other languages. The overall mean reference line helps identify which languages are above or below average.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(BG)

avg_dur = master_df.groupby('language')['duration_seconds'].mean().reindex(LANGUAGES)
bars = ax.bar([LANG_LABELS[l] for l in avg_dur.index],
              avg_dur.values,
              color=COLOR_LIST, edgecolor='white', linewidth=0.5, width=0.6)

for bar, val in zip(bars, avg_dur.values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.1,
            f'{val:.2f}s', ha='center', va='bottom',
            fontsize=10, fontweight='bold', color=TEXT)

ax.axhline(avg_dur.mean(), color='white', linestyle='--',
           linewidth=1.5, label=f'Overall mean: {avg_dur.mean():.2f}s')

ax.set_title('Bar Chart Showing Average Audio Duration per Language',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Language', labelpad=10)
ax.set_ylabel('Average Duration (seconds)', labelpad=10)
ax.tick_params(axis='x', rotation=25)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.4)
ax.set_axisbelow(True)
ax.text(0.01, 0.97, 'MULTIVARIATE Comparison',
        transform=ax.transAxes, fontsize=9, color=SUBTEXT, va='top')

plt.tight_layout()
save_show(fig, 'avg_duration.png')

5. Stacked Bargraph showing Split Proportions per Language

Shows train, dev, and test as percentages stacked to 100% for each language. It is important for confirming that every language has a consistent and fair split ratio. If one language has a much smaller test proportion its evaluation metrics will be less trustworthy.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor(BG)

split_lang = master_df.groupby(['language', 'split']).size().unstack(fill_value=0)
split_lang  = split_lang.reindex(LANGUAGES)
split_pct   = split_lang.div(split_lang.sum(axis=1), axis=0) * 100
lang_names  = [LANG_LABELS[l] for l in split_pct.index]

split_cols = {'train': '#4CAF50', 'dev': '#FF9800', 'test': '#E63946'}
bottoms    = np.zeros(len(split_pct))

for split, color in split_cols.items():
    if split in split_pct.columns:
        vals = split_pct[split].values
        bars = ax.bar(lang_names, vals, bottom=bottoms,
                      label=split.capitalize(),
                      color=color, edgecolor='white', linewidth=0.3)
        for bar, val, bot in zip(bars, vals, bottoms):
            if val > 3:
                ax.text(bar.get_x() + bar.get_width()/2,
                        bot + val/2,
                        f'{val:.1f}%',
                        ha='center', va='center',
                        fontsize=9, color='white', fontweight='bold')
        bottoms += vals

ax.set_title('Stacked Bargraph Showing Split Proportions per Language',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Language', labelpad=10)
ax.set_ylabel('Percentage (%)', labelpad=10)
ax.tick_params(axis='x', rotation=25)
ax.legend(fontsize=10, loc='lower right')
ax.set_ylim(0, 105)
ax.text(0.01, 0.97, 'MULTIVARIATE Comparison',
        transform=ax.transAxes, fontsize=9, color=SUBTEXT, va='top')

plt.tight_layout()
save_show(fig, 'split_proportions.png')

6.  Count Plot of Speaker per Language

Shows how many samples each speaker contributed per language using color coded bars. Important for detecting speaker dominance — if one speaker recorded 90% of samples the model learns that speaker's voice style disproportionately. For your dataset English is the only language with multiple speakers.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(BG)

temp_df = master_df.copy()
temp_df['lang_label'] = temp_df['language'].map(LANG_LABELS)

sns.countplot(data=temp_df, x='lang_label',
              order=LABEL_LIST,
              hue='speaker_id',
              edgecolor='white', linewidth=0.4, ax=ax)

ax.set_title('Count Plot of Sample Count per Speaker per Language',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Language', labelpad=10)
ax.set_ylabel('Count', labelpad=10)
ax.tick_params(axis='x', rotation=25)
ax.legend(title='Speaker ID', fontsize=8, title_fontsize=9)
ax.grid(axis='y', alpha=0.4)
ax.set_axisbelow(True)
ax.text(0.01, 0.97, 'MULTIVARIATE Count Plot',
        transform=ax.transAxes, fontsize=9, color=SUBTEXT, va='top')

plt.tight_layout()
save_show(fig, 'speaker_countplot.png')

7. Heatmap of Feature Correlation

Shows the correlation coefficient between duration and word count as a color coded grid. Confirms the relationship we found earlier 0.33 correlation meaning longer texts tend to produce longer audio but the relationship is only moderate.
It is important for deciding whether to use both features or just one in your model.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
fig.patch.set_facecolor(BG)

corr = master_df[['duration_seconds', 'word_count']].corr()

sns.heatmap(corr, annot=True, fmt='.3f',
            cmap='coolwarm', ax=ax,
            linewidths=1, linecolor=BG,
            annot_kws={'size': 14, 'weight': 'bold'},
            vmin=-1, vmax=1,
            xticklabels=['Duration', 'Word Count'],
            yticklabels=['Duration', 'Word Count'])

ax.set_title('Feature Correlation Heatmap',
             fontsize=15, fontweight='bold', pad=15)
ax.text(0.01, 0.01, 'MULTIVARIATE Heatmap',
        transform=ax.transAxes, fontsize=9, color=SUBTEXT)

plt.tight_layout()
save_show(fig, 'correlation_heatmap.png')

8. Heatmap of Sample Count Language vs Split

Shows exact sample counts for every language and split combination in one view. It is useful for quickly spotting any language that is severely underrepresented in a specific split. Lugbara has the fewest samples across all splits.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
fig.patch.set_facecolor(BG)

pivot = master_df.groupby(['language', 'split']).size().unstack(fill_value=0)
pivot.index = [LANG_LABELS[l] for l in pivot.index]

sns.heatmap(pivot, annot=True, fmt='d',
            cmap='YlOrRd', ax=ax,
            linewidths=0.8, linecolor=BG,
            annot_kws={'size': 12, 'weight': 'bold', 'color': 'black'})

ax.set_title('Heatmap of Sample Count: Language vs Split',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Split', labelpad=10)
ax.set_ylabel('Language', labelpad=10)
ax.text(0.01, 0.01, 'MULTIVARIATE Heatmap',
        transform=ax.transAxes, fontsize=9, color=SUBTEXT)

plt.tight_layout()
save_show(fig, 'sample_heatmap.png')

9.  Heatmap of Speaker Frequency per Language

Shows how many samples each speaker ID recorded per language. Reveals whether any single speaker dominates the dataset. Important because speaker diversity affects how natural and generalizable the synthesized voices will sound.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor(BG)

spk_pivot = master_df.groupby(['language', 'speaker_id']).size().unstack(fill_value=0)
spk_pivot.index = [LANG_LABELS[l] for l in spk_pivot.index]

sns.heatmap(spk_pivot, annot=True, fmt='d',
            cmap='Purples', ax=ax,
            linewidths=0.8, linecolor=BG,
            annot_kws={'size': 11, 'weight': 'bold', 'color': 'black'})

ax.set_title('Heatmap of Speaker Frequency per Language',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Speaker ID', labelpad=10)
ax.set_ylabel('Language', labelpad=10)
ax.text(0.01, 0.01, 'MULTIVARIATE Heatmap',
        transform=ax.transAxes, fontsize=9, color=SUBTEXT)

plt.tight_layout()
save_show(fig, 'speaker_heatmap.png')

10. Heatmap of Language vs Audio Quality Bins

Groups audio files into duration bins (under 2s, 2 to 5s, 5 to 10s, 10 to 20s, over 20s) and shows the count per language.
It is important because it quantifies exactly how many Lugbara recordings fall in the very long bin versus other languages. Helps decide whether to cap or segment long audio files before training.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor(BG)

# Bin duration into quality categories
bins   = [0, 2, 5, 10, 20, 60]
labels = ['<2s', '2-5s', '5-10s', '10-20s', '>20s']
master_df['duration_bin'] = pd.cut(
    master_df['duration_seconds'],
    bins=bins, labels=labels)

quality_pivot = master_df.groupby(
    ['language', 'duration_bin']).size().unstack(fill_value=0)
quality_pivot.index = [LANG_LABELS[l] for l in quality_pivot.index]

sns.heatmap(quality_pivot, annot=True, fmt='d',
            cmap='Blues', ax=ax,
            linewidths=0.8, linecolor=BG,
            annot_kws={'size': 11, 'weight': 'bold', 'color': 'black'})

ax.set_title('Heatmap of Language vs Audio Duration Quality Bins',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Duration Bin', labelpad=10)
ax.set_ylabel('Language', labelpad=10)
ax.text(0.01, 0.01, 'MULTIVARIATE Heatmap',
        transform=ax.transAxes, fontsize=9, color=SUBTEXT)

plt.tight_layout()
save_show(fig, 'quality_bins_heatmap.png')

11. Heatmap of Missing Data

Shows a visual map of where null values exist across features and samples. Even though the dataset has zero missing values this plot is important as a verification step and for any future data additions. A completely dark heatmap with no red confirms the dataset is complete.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor(BG)

cols_to_check = ['duration_seconds', 'word_count', 'speaker_id', 'text', 'split']
missing       = master_df[cols_to_check].isnull().astype(int)

sns.heatmap(missing.T,
            cmap=sns.color_palette(['#1A1D27', '#E63946'], as_cmap=True),
            ax=ax, cbar_kws={'label': 'Missing (1=Yes, 0=No)'},
            yticklabels=cols_to_check,
            xticklabels=False)

ax.set_title('Missing Data Heatmap',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Sample Index', labelpad=10)
ax.text(0.01, 0.01, 'MULTIVARIATE Heatmap',
        transform=ax.transAxes, fontsize=9, color=SUBTEXT)

plt.tight_layout()
save_show(fig, 'missing_data_heatmap.png')

12. Waveform: One Sample per Language

Plots the raw audio signal amplitude over time for one recording per language. Important for visually confirming that recordings are real speech and not silence or noise.
For this dataset it helps compare speaking pace and loudness patterns across the 7 languages.

In [ ]:
fig, axes = plt.subplots(len(LANGUAGES), 1, figsize=(14, 16))
fig.patch.set_facecolor(BG)
fig.suptitle('Waveforms per Language',
             fontsize=16, fontweight='bold', y=1.01)

for ax, lang in zip(axes, LANGUAGES):
    sample = master_df[master_df['language'] == lang].iloc[0]
    data, sr = decode_audio(sample['audio'])
    times    = np.linspace(0, len(data)/sr, len(data))
    ax.plot(times, data, color=LANG_COLORS[lang], linewidth=0.5)
    ax.fill_between(times, data, alpha=0.2, color=LANG_COLORS[lang])
    ax.set_ylabel(LANG_LABELS[lang], fontsize=9)
    ax.set_xlim(0, times[-1])
    ax.grid(alpha=0.3)
    ax.set_facecolor(PANEL)

axes[-1].set_xlabel('Time (seconds)', labelpad=10)
fig.text(0.01, 0.01, 'MULTIVARIATE Audio/Signal',
         fontsize=9, color=SUBTEXT)

plt.tight_layout()
save_show(fig, 'waveforms.png')

13. Spectrogram: One Sample per Language

Shows frequency content over time for one recording per language using color intensity. It reveals the frequency fingerprint of each language. High frequency consonants and low frequency vowels appear as distinct patterns making it possible to visually distinguish language characteristics.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
fig.patch.set_facecolor(BG)
fig.suptitle('Spectrograms per Language',
             fontsize=16, fontweight='bold')

axes_flat = axes.flatten()

for i, lang in enumerate(LANGUAGES):
    ax   = axes_flat[i]
    sample = master_df[master_df['language'] == lang].iloc[0]
    data, sr = decode_audio(sample['audio'])
    D    = librosa.amplitude_to_db(np.abs(librosa.stft(data)), ref=np.max)
    librosa.display.specshow(D, sr=sr, x_axis='time',
                             y_axis='hz', ax=ax, cmap='magma')
    ax.set_title(LANG_LABELS[lang], fontsize=11, fontweight='bold')
    ax.set_facecolor(PANEL)

# Hide unused subplot
axes_flat[-1].set_visible(False)
fig.text(0.01, 0.01, 'MULTIVARIATE Audio/Signal',
         fontsize=9, color=SUBTEXT)

plt.tight_layout()
save_show(fig, 'spectrograms.png')

14. MFCC Plot per Language

Shows Mel Frequency Cepstral Coefficients which are the features most commonly used in speech recognition and voice synthesis models. MFCCs are what the model actually learns from. Different patterns per language confirm the languages are acoustically distinct enough for classification.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
fig.patch.set_facecolor(BG)
fig.suptitle('MFCC per Language',
             fontsize=16, fontweight='bold')

axes_flat = axes.flatten()

for i, lang in enumerate(LANGUAGES):
    ax     = axes_flat[i]
    sample = master_df[master_df['language'] == lang].iloc[0]
    data, sr = decode_audio(sample['audio'])
    mfccs  = librosa.feature.mfcc(y=data, sr=sr, n_mfcc=13)
    librosa.display.specshow(mfccs, sr=sr, x_axis='time', ax=ax, cmap='coolwarm')
    ax.set_title(LANG_LABELS[lang], fontsize=11, fontweight='bold')
    ax.set_ylabel('MFCC Coefficients')
    ax.set_facecolor(PANEL)

axes_flat[-1].set_visible(False)
fig.text(0.01, 0.01, 'MULTIVARIATE Audio/Signal',
         fontsize=9, color=SUBTEXT)

plt.tight_layout()
save_show(fig, 'mfcc.png')

15. Pitch Distribution per Language

Shows the pitch frequency distribution for each language. Important for voice agent development because pitch patterns differ significantly across languages and speakers. Tonal languages show wider pitch ranges. For this dataset it helps characterize the acoustic properties that the voice selection model needs to learn.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor(BG)

for lang in LANGUAGES:
    sample   = master_df[master_df['language'] == lang].iloc[0]
    data, sr = decode_audio(sample['audio'])
    pitches, magnitudes = librosa.piptrack(y=data, sr=sr)
    pitch_vals = pitches[pitches > 0]
    if len(pitch_vals) > 0:
        pd.Series(pitch_vals).plot.kde(
            ax=ax, label=LANG_LABELS[lang],
            color=LANG_COLORS[lang], linewidth=2)

ax.set_title('Pitch Distribution per Language',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Pitch (Hz)', labelpad=10)
ax.set_ylabel('Density', labelpad=10)
ax.set_xlim(0, 1000)
ax.legend(fontsize=9)
ax.grid(alpha=0.4)
ax.set_axisbelow(True)
ax.text(0.01, 0.97, 'MULTIVARIATE Audio/Signal',
        transform=ax.transAxes, fontsize=9, color=SUBTEXT, va='top')

plt.tight_layout()
save_show(fig, 'pitch_distribution.png')

16. Energy/Amplitude Density per Language

Shows a smooth density curve of RMS energy values per language overlaid on one chart. Useful for comparing recording loudness consistency across languages. If one language has a very different energy profile it may need audio normalization before training to prevent the model from using volume as a language cue.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor(BG)

for lang in LANGUAGES:
    sample   = master_df[master_df['language'] == lang].iloc[0]
    data, sr = decode_audio(sample['audio'])
    rms      = librosa.feature.rms(y=data)[0]
    pd.Series(rms).plot.kde(
        ax=ax, label=LANG_LABELS[lang],
        color=LANG_COLORS[lang], linewidth=2)

ax.set_title('Energy (RMS Amplitude) Density per Language',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('RMS Energy', labelpad=10)
ax.set_ylabel('Density', labelpad=10)
ax.legend(fontsize=9)
ax.grid(alpha=0.4)
ax.set_axisbelow(True)
ax.text(0.01, 0.97, 'MULTIVARIATE Audio/Signal',
        transform=ax.transAxes, fontsize=9, color=SUBTEXT, va='top')

plt.tight_layout()
save_show(fig, 'energy_density.png')

17.  PCA Scatter Plot

Reduces the 13 MFCC features down to 2 dimensions and plots each sample as a dot colored by language. Critical for understanding whether languages are acoustically separable. If language clusters are well separated in PCA space the voice assistant can reliably distinguish them. Overlapping clusters indicate languages that sound acoustically similar and may be harder to classify.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
fig.patch.set_facecolor(BG)

# Extract MFCC features for a sample per language
print("Extracting audio features for PCA...")
features, lang_tags = [], []

for lang in LANGUAGES:
    subset = master_df[master_df['language'] == lang].head(100)
    for _, row in subset.iterrows():
        try:
            data, sr = decode_audio(row['audio'])
            mfcc     = librosa.feature.mfcc(y=data, sr=sr, n_mfcc=13)
            features.append(mfcc.mean(axis=1))
            lang_tags.append(lang)
        except:
            pass

X      = np.array(features)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca    = PCA(n_components=2, random_state=42)
X_pca  = pca.fit_transform(X_scaled)

for lang in LANGUAGES:
    mask = np.array(lang_tags) == lang
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               color=LANG_COLORS[lang], label=LANG_LABELS[lang],
               alpha=0.6, s=40, edgecolors='white', linewidth=0.3)

ax.set_title('PCA of Audio Features (MFCC) by Language',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', labelpad=10)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', labelpad=10)
ax.legend(fontsize=9)
ax.grid(alpha=0.4)
ax.set_axisbelow(True)
ax.text(0.01, 0.97, 'MULTIVARIATE Advanced',
        transform=ax.transAxes, fontsize=9, color=SUBTEXT, va='top')

plt.tight_layout()
save_show(fig, 'pca.png')

18.  t-SNE Plot

Audio Embeddings by Language
Similar to PCA but uses a non-linear algorithm that better preserves local cluster structure. Reveals tighter and more distinct clusters than PCA. For the voice assistant, it is important advanced visualizations because it shows whether a model can realistically separate all 7 languages based on audio features alone.

In [ ]:
# Reuse X_scaled and lang_tags from Cell 17
fig, ax = plt.subplots(figsize=(11, 8))
fig.patch.set_facecolor(BG)

print("Running t-SNE...")
tsne    = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_tsne  = tsne.fit_transform(X_scaled)

for lang in LANGUAGES:
    mask = np.array(lang_tags) == lang
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
               color=LANG_COLORS[lang], label=LANG_LABELS[lang],
               alpha=0.6, s=40, edgecolors='white', linewidth=0.3)

ax.set_title('t-SNE of Audio Embeddings (MFCC) by Language',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('t-SNE Dimension 1', labelpad=10)
ax.set_ylabel('t-SNE Dimension 2', labelpad=10)
ax.legend(fontsize=9)
ax.grid(alpha=0.4)
ax.set_axisbelow(True)
ax.text(0.01, 0.97, 'MULTIVARIATE Advanced',
        transform=ax.transAxes, fontsize=9, color=SUBTEXT, va='top')

plt.tight_layout()
save_show(fig, 'tsne.png')

19. Word Clouds per Language

Shows the most frequent words in each language's transcriptions as a visual cloud where larger words appear more often. its useful for understanding vocabulary distribution and confirming the parallel corpus nature of the dataset. If the same concepts dominate all languages it confirms the translations are consistent across the dataset.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.patch.set_facecolor(BG)
fig.suptitle('Word Clouds per Language',
             fontsize=16, fontweight='bold')

axes_flat = axes.flatten()

for i, lang in enumerate(LANGUAGES):
    ax      = axes_flat[i]
    texts   = ' '.join(master_df[master_df['language'] == lang]['text'].astype(str).tolist())
    wc      = WordCloud(width=400, height=300,
                        background_color=PANEL,
                        colormap='plasma',
                        max_words=80,
                        prefer_horizontal=0.8).generate(texts)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(LANG_LABELS[lang], fontsize=12, fontweight='bold', color=TEXT)

axes_flat[-1].set_visible(False)
fig.text(0.01, 0.01, 'MULTIVARIATE Advanced',
         fontsize=9, color=SUBTEXT)

plt.tight_layout()
save_show(fig, 'wordclouds.png')

20. Scatter Plot of Duration vs Word Count

Plots every sample as a dot with word count on x axis and duration on y axis colored by language. The trend line shows the overall relationship. It is important for confirming that longer sentences produce proportionally longer audio across all languages and for identifying any language that breaks this pattern unexpectedly.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
fig.patch.set_facecolor(BG)

for lang in LANGUAGES:
    subset = master_df[master_df['language'] == lang]
    ax.scatter(subset['word_count'], subset['duration_seconds'],
               color=LANG_COLORS[lang], label=LANG_LABELS[lang],
               alpha=0.25, s=15, edgecolors='none')

# Trend line
x_all = master_df['word_count']
y_all = master_df['duration_seconds']
z     = np.polyfit(x_all, y_all, 1)
p     = np.poly1d(z)
x_line = np.linspace(x_all.min(), x_all.max(), 200)
ax.plot(x_line, p(x_line), color='white',
        linewidth=2, linestyle='--', label='Trend line')

ax.set_title('Scatter Plot of Duration vs Word Count by Language',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Word Count', labelpad=10)
ax.set_ylabel('Duration (seconds)', labelpad=10)
ax.legend(fontsize=8, markerscale=2)
ax.grid(alpha=0.4)
ax.set_axisbelow(True)
ax.text(0.01, 0.97, 'MULTIVARIATE Advanced',
        transform=ax.transAxes, fontsize=9, color=SUBTEXT, va='top')

plt.tight_layout()
save_show(fig, 'duration_vs_wordcount.png')

21. KDE Word Count Density per Language

 Smooth density curves of word count for all 7 languages on one chart. The best visualization for directly comparing sentence length tendencies across languages. For this dataset it clearly shows that Lugbara has a higher peak at longer word counts while Runyankole and Luganda peak at shorter sentences.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor(BG)

for lang in LANGUAGES:
    subset = master_df[master_df['language'] == lang]['word_count']
    subset.plot.kde(ax=ax, color=LANG_COLORS[lang],
                    linewidth=2.5, label=LANG_LABELS[lang])

ax.set_title('Word Count KDE Density per Language',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Word Count', labelpad=10)
ax.set_ylabel('Density', labelpad=10)
ax.legend(fontsize=9)
ax.set_xlim(0, master_df['word_count'].quantile(0.99))
ax.grid(alpha=0.4)
ax.set_axisbelow(True)
ax.text(0.01, 0.97, 'MULTIVARIATE Advanced',
        transform=ax.transAxes, fontsize=9, color=SUBTEXT, va='top')

plt.tight_layout()
save_show(fig, 'wordcount_kde.png')

Ydata Profiling for Reports

In [ ]:
df_profile = master_df.drop(columns=['audio'])

profile = ProfileReport(
    df_profile,
    title="SUNBIRD SALT Dataset EDA Report",
    explorative=True,

)

# Save as HTML
profile.to_file("/mnt/user-data/outputs/salt_eda_report.html")
print("Profile report saved!")

# Display inline
profile.to_notebook_iframe()

References:

[1] A Data Scientist's Essential Guide to Exploratory Data Analysis       
    Towards Data Science

    Links                                   
    1. https://towardsdatascience.com   
    2. https://towardsdatascience.com/a-data-scientists-essential-guide-to-exploratory-data-analysis-25637eee0cf6/                       
                                                             
[2] Sunbird AI — SALT Dataset                               
     https://huggingface.co/datasets/sunbird/salt   

[3] Geeks for Geeks
  1. https://www.geeksforgeeks.org/nlp/mel-frequency-cepstral-coefficients-mfcc-for-speech-recognition/

AI Tools Used: claude.ai
